In [4]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

In [5]:
DATA = Path("../data/processed")
train = pd.read_parquet(DATA / "train.parquet")
test = pd.read_parquet(DATA / "test.parquet")
print("Train:", train.shape, "Test:", test.shape)

Train: (26064, 12) Test: (6517, 12)


In [8]:

NUMERIC_FEATURES = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "cb_person_cred_hist_length",
    "loan_to_income_ratio",
    "log_income",
    "log_loan_amount",
]

CATEGORICAL_FEATURES = [
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file",
]

INPUT_FEATURES = [
    "person_age",
    "person_income",
    "person_home_ownership",
    "person_emp_length",
    "loan_intent",
    "loan_amnt",
    "cb_person_default_on_file",
    "cb_person_cred_hist_length",
]

In [6]:
def engineer_features(df):
    df = df.copy()

    # 1. handle implausible ages.
    df.loc[
        ~df["person_age"].between(18, 100),
        "person_age"
    ] = np.nan

    # 2. handle implausible employment lengths.
    invalid_employment = (
        (df["person_emp_length"] < 0)
        | (df["person_emp_length"] > df["person_age"] - 16)
    )

    df.loc[invalid_employment, "person_emp_length"] = np.nan

    # 3. create loan-to-income ratio.
    df["loan_to_income_ratio"] = (
        df["loan_amnt"] / df["person_income"].replace(0, np.nan)
    )

    # 4. create log-transformed features.
    df["log_income"] = np.log1p(
        df["person_income"].clip(lower=0)
    )

    df["log_loan_amount"] = np.log1p(
        df["loan_amnt"].clip(lower=0)
    )

    return df

In [10]:
train_fe = engineer_features(train)
test_fe = engineer_features(test)
print(train_fe[["person_age", "person_emp_length", "loan_to_income_ratio"]].isna().sum())

person_age                 5
person_emp_length       1328
loan_to_income_ratio       0
dtype: int64


In [ ]:
train_fe.to_parquet(DATA / "train_fe.parquet", index=False)
test_fe.to_parquet(DATA / "test_fe.parquet", index=False)
print("Saved train_fe.parquet and test_fe.parquet")

Saved train_fe.parquet and test_fe.parquet


: 